# Import Module

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import warnings
from datetime import date
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import NotFound
warnings.filterwarnings("ignore")

forecast_date = pd.Timestamp.today().date()

pd.reset_option('display.max_columns')

# Setup BQ Connection

Autentikasi BigQuery menggunakan service account yang dikonfigurasi melalui Application Default Credentials (ADC) di environment resmi. File kredensial tidak disimpan di notebook.

In [ ]:
# BigQuery authentication uses the service account configured through ADC.
# Manage credentials outside this notebook; do not commit credential files.


# Penarikan Data Source

## 1. Retention Rate (RR)

In [ ]:
# Query penarikan kolom-kolom relevan saja
query_RR = """
SELECT
    months,
    channel,
    productcategory,
    SAFE_DIVIDE(
        cust_trx_repurchase_success,
        cust_trx_success
    ) AS repurchase_rate
FROM your-project.your_dataset.portfolio_table_01
WHERE LOWER(product) = 'all'
  AND months < DATE_TRUNC(CURRENT_DATE(), MONTH)
  AND category = 'ACCUMULATIVE'

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        months,
        channel,
        productcategory
    ORDER BY last_refresh DESC
) = 1

ORDER BY
    months,
    channel,
    productcategory
"""

# Run query and load into a pandas DataFrame
df_RR = client.query(query_RR).to_dataframe()
df_RR

## 2. Customer Value (CV)

In [ ]:
# Query penarikan relevan data saja
query_CV = """
SELECT
    months,
    category,
    channel,
    productcategory,
    SAFE_DIVIDE(value, cust) AS customer_value
FROM your-project.your_dataset.portfolio_table_02
WHERE LOWER(product) = 'all'
  AND months < DATE_TRUNC(CURRENT_DATE(), MONTH)

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        months,
        category,
        channel,
        productcategory
    ORDER BY last_refresh DESC
) = 1

ORDER BY
    months,
    category,
    channel,
    productcategory
"""

# Run query and load into a pandas DataFrame
df_CV = client.query(query_CV).to_dataframe()
df_CV

## 3. Linkage Rate (LR)

In [ ]:
# Query penarikan kolom-kolom relevan saja
query_LR = """
SELECT
    period,
    pair,
    SAFE_DIVIDE(leads, base) AS linkage_rate
FROM your-project.your_dataset.portfolio_table_03
WHERE period < DATE_TRUNC(CURRENT_DATE(), MONTH)

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        period,
        pair
    ORDER BY last_refersh DESC
) = 1

ORDER BY
    period,
    pair
"""

# Run query and load into a pandas DataFrame
df_LR = client.query(query_LR).to_dataframe()
df_LR

## 4. NCLCO

In [ ]:
query_nclco = """
SELECT
    bulan,
    channel,
    productcategory,
    nc,
    lco,
    lc
FROM your-project.your_dataset.portfolio_table_04
WHERE LOWER(category) = 'by month'
  AND LOWER(product) = 'all'
  AND bulan < DATE_TRUNC(CURRENT_DATE(), MONTH)

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        bulan,
        channel,
        productcategory
    ORDER BY last_refresh DESC
) = 1

ORDER BY
    bulan,
    channel,
    productcategory
"""

# Run query and load into a pandas DataFrame
df_nclco = client.query(query_nclco).to_dataframe()
df_nclco

# Logic for Forecasting

## 1. Retention Rate (RR)

In [ ]:
df_RR['months'] = pd.to_datetime(df_RR['months'])
df_RR = df_RR.sort_values('months')

def project_rr(group):

    group = (
        group
        .sort_values('months')
        .dropna(subset=['repurchase_rate'])
        .copy()
    )

    if len(group) == 0:
        return None

    current = group['repurchase_rate'].iloc[-1]

    # pakai histori 18 bulan terakhir
    recent_history = group.tail(18)

    peak = recent_history['repurchase_rate'].max()

    # jika current sudah peak
    peak = max(peak, current)

    gap = peak - current

    history_length = len(group)

    if history_length < 6:
        RECOVERY_FACTOR = 0.30

    elif history_length < 12:
        RECOVERY_FACTOR = 0.50

    else:
        RECOVERY_FACTOR = 0.60

    target_dec = current + (
        gap * RECOVERY_FACTOR
    )

    today = pd.Timestamp.today()

    forecast_start = (
        today
        .to_period('M')
        .to_timestamp()
    )

    forecast_end = pd.Timestamp(
        year=today.year,
        month=12,
        day=1
    )

    future_months = pd.date_range(
        start=forecast_start,
        end=forecast_end,
        freq='MS'
    )

    projection = np.linspace(
        current,
        target_dec,
        len(future_months) + 1
    )[1:]

    return pd.DataFrame({
        'months': future_months,
        'channel': group['channel'].iloc[0],
        'productcategory': group['productcategory'].iloc[0],
        'repurchase_rate_projection': projection
    })

results = []

for _, group in df_RR.groupby(
    ['channel','productcategory']
):

    proj = project_rr(group)

    if proj is not None:
        results.append(proj)

projection_rr = pd.concat(
    results,
    ignore_index=True
)

projection_rr.head(10)

## 2. Customer Value (CV)

In [ ]:
def project_cv(group):

    group = (
        group
        .sort_values('months')
        .dropna(subset=['customer_value'])
        .copy()
    )

    if len(group) == 0:
        return None

    history_length = len(group)

    current = group['customer_value'].iloc[-1]

    today = pd.Timestamp.today()

    forecast_start = (
        today
        .to_period('M')
        .to_timestamp()
    )

    forecast_end = pd.Timestamp(
        year=today.year,
        month=12,
        day=1
    )

    future_months = pd.date_range(
        start=forecast_start,
        end=forecast_end,
        freq='MS'
    )

    # =====================
    # TIER A
    # =====================
    if history_length >= 6:

        recent = group.tail(6)

        first = recent['customer_value'].iloc[0]
        last = recent['customer_value'].iloc[-1]

        if first > 0:

            monthly_growth = (
                (last / first) **
                (1 / (len(recent) - 1))
            ) - 1

        else:

            monthly_growth = 0.01

    # =====================
    # TIER B
    # =====================
    elif history_length >= 3:

        recent = group

        growth_rates = []

        for i in range(1, len(recent)):

            prev = recent['customer_value'].iloc[i-1]
            curr = recent['customer_value'].iloc[i]

            if prev > 0:

                growth_rates.append(
                    (curr / prev) - 1
                )

        if len(growth_rates) > 0:

            monthly_growth = np.mean(
                growth_rates
            )

        else:

            monthly_growth = 0.01

    # =====================
    # TIER C
    # =====================
    else:

        monthly_growth = 0.01

    # =====================
    # BUSINESS CONSTRAINT
    # =====================

    monthly_growth = np.clip(
        monthly_growth,
        0.005,   # minimum +0.5%
        0.03     # maximum +3%
    )

    projection = []

    value = current

    for _ in range(len(future_months)):

        value = value * (
            1 + monthly_growth
        )

        projection.append(value)

    return pd.DataFrame({
        'months': future_months,
        'category': group['category'].iloc[0],
        'channel': group['channel'].iloc[0],
        'productcategory': group['productcategory'].iloc[0],
        'customer_value_projection': projection,
        # 'monthly_growth_used': monthly_growth
    })

results = []

for _, group in df_CV.groupby(
    ['category','channel','productcategory']
):

    proj = project_cv(group)

    if proj is not None:
        results.append(proj)

projection_cv = pd.concat(
    results,
    ignore_index=True
)

projection_cv.head(14)

## 3. Linkage Rate (LR)

In [ ]:
df_LR['period'] = pd.to_datetime(df_LR['period'])
df_LR = df_LR.sort_values('period')

def project_lr(group):

    group = (
        group
        .sort_values('period')
        .dropna(subset=['linkage_rate'])
        .copy()
    )

    if len(group) == 0:
        return None

    current = group['linkage_rate'].iloc[-1]

    # seluruh histori (karena baru mulai Jan-2025)
    recent_history = group

    peak = (recent_history['linkage_rate'].nlargest(3).mean())

    peak = max(peak, current)

    gap = peak - current

    history_length = len(group)

    if history_length < 6:
        recovery_factor = 0.30

    elif history_length < 12:
        recovery_factor = 0.50

    else:
        recovery_factor = 0.60

    target_dec = current + (
        gap * recovery_factor
    )

    today = pd.Timestamp.today()

    forecast_start = (
        today
        .to_period('M')
        .to_timestamp()
    )

    forecast_end = pd.Timestamp(
        year=today.year,
        month=12,
        day=1
    )

    future_months = pd.date_range(
        start=forecast_start,
        end=forecast_end,
        freq='MS'
    )

    projection = np.linspace(
        current,
        target_dec,
        len(future_months) + 1
    )[1:]

    # safety untuk rate
    projection = np.clip(
        projection,
        0,
        1
    )

    return pd.DataFrame({
        'period': future_months,
        'pair': group['pair'].iloc[0],
        'linkage_rate_projection': projection
    })


results = []

for _, group in df_LR.groupby(
    ['pair']
):

    proj = project_lr(group)

    if proj is not None:
        results.append(proj)

projection_lr = pd.concat(
    results,
    ignore_index=True
)

projection_lr.head(20)

## 4. NCLCO

## NCLCO forecast logic (omitted)

The original workflow uses company-specific product and channel hierarchies. Those mappings and the associated implementation are excluded from this public portfolio copy.


# Append Data Forecast to Actual

## 1. Repurchase Rate (RR)

In [ ]:
rr_forecast = (
    projection_rr
    .rename(
        columns={
            'repurchase_rate_projection':
            'repurchase_rate'
        }
    )
)

rr_forecast['data_type'] = 'FORECAST'

rr_actual = df_RR.copy()
rr_actual['data_type'] = 'ACTUAL'

rr_final = pd.concat(
    [
        rr_actual,
        rr_forecast
    ],
    ignore_index=True
)

rr_final['tanggal_forecast'] = forecast_date
periode_forecast_rr = projection_rr["months"].min()

rr_final["periode_forecast"] = periode_forecast_rr
rr_final

## 2. Customer Value (CV)

In [ ]:
cv_forecast = (
    projection_cv
    .rename(
        columns={
            'customer_value_projection':
            'customer_value'
        }
    )
)

cv_forecast['data_type'] = 'FORECAST'

cv_actual = df_CV.copy()
cv_actual['data_type'] = 'ACTUAL'

cv_final = pd.concat(
    [
        cv_actual,
        cv_forecast
    ],
    ignore_index=True
)

cv_final['tanggal_forecast'] = forecast_date
periode_forecast_cv = projection_cv["months"].min()

cv_final["periode_forecast"] = periode_forecast_cv
cv_final

## 3. Linkage Rate (LR)

In [ ]:
lr_forecast = (
    projection_lr
    .rename(
        columns={
            'linkage_rate_projection':
            'linkage_rate'
        }
    )
)

lr_forecast['data_type'] = 'FORECAST'

lr_actual = df_LR.copy()
lr_actual['data_type'] = 'ACTUAL'

lr_final = pd.concat(
    [
        lr_actual,
        lr_forecast
    ],
    ignore_index=True
)

lr_final['tanggal_forecast'] = forecast_date

periode_forecast_lr = projection_lr["period"].min()

lr_final["periode_forecast"] = periode_forecast_lr
lr_final

## 4. NCLCO

NCLCO-specific forecast-to-actual append logic is omitted from this public copy because it depends on the internal hierarchy and source schema.


# Create/Update Table in BQ

In [ ]:
# Uses the service account configured through Application Default Credentials.
client = bigquery.Client()
print("Connected to BigQuery using the configured service account.")


In [ ]:
def upload_forecast(df, table_id):

    periode = pd.to_datetime(
        df["periode_forecast"].iloc[0]
    )

    # =========================================================
    # 1. CEK APAKAH TABLE SUDAH ADA
    # =========================================================

    try:
        client.get_table(table_id)
        table_exists = True

    except NotFound:
        table_exists = False

    # =========================================================
    # 2. KALAU TABLE BELUM ADA → CREATE + UPLOAD
    # =========================================================

    if not table_exists:
        print(f"TABLE NOT FOUND : {table_id}")
        print(f"Creating table and uploading {len(df):,} rows...")

        job = client.load_table_from_dataframe(df, table_id,
            job_config=bigquery.LoadJobConfig(
                write_disposition="WRITE_EMPTY"
            )
        )

        job.result()

        print(
            f"SUCCESS : {table_id} | "
            f"created + rows={len(df):,}"
        )

        return

    # =========================================================
    # 3. KALAU TABLE SUDAH ADA → CEK PERIODE
    # =========================================================

    query = f"""
    SELECT COUNT(*) AS cnt
    FROM `{table_id}`
    WHERE periode_forecast = @periode
    """

    job_config = bigquery.QueryJobConfig(

        query_parameters=[
            bigquery.ScalarQueryParameter(
                "periode",
                "DATE",
                periode.date()
            )
        ]
    )

    exists = (
        client.query(
            query,
            job_config=job_config
        )
        .result()
        .to_dataframe()
        .iloc[0]["cnt"]

    )

    # =========================================================
    # 4. PERIODE SUDAH ADA → SKIP
    # =========================================================

    if exists > 0:
        print(f"SKIP : {table_id}")
        print(f"Forecast periode {periode.date()} sudah ada.")

        return

    # =========================================================
    # 5. PERIODE BELUM ADA → APPEND
    # =========================================================

    job = client.load_table_from_dataframe(df, table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        )
    )

    job.result()

    print(f"SUCCESS : {table_id} | rows={len(df):,}")

# Create/Update Forecast Table
upload_forecast(
    rr_final,
    "your-project.your_dataset.portfolio_table_05"
)

upload_forecast(
    cv_final,
    "your-project.your_dataset.portfolio_table_06"
)

upload_forecast(
    lr_final,
    "your-project.your_dataset.portfolio_table_07"
)

upload_forecast(
    nclco_final,
    "your-project.your_dataset.portfolio_table_08"
)

#NCLCO VALIDASI

## Authorized actual-data adapter (omitted)

The original SQL Server connection details, source query, account name, network address, and internal schema have been removed. Supply the required historical actuals through an approved data-access adapter in your authorized environment. No sample or dummy data is provided.


## NCLCO allocation and persistence validation (omitted)

Company-specific allocation, database integration, table schema, and reconciliation implementation are excluded from this public copy. The generic BigQuery authentication and upload pattern is shown in the preceding section.


Brand-specific historical diagnostics are omitted from this public copy.


## Rolling-window allocation backtest (omitted)

The source notebook compares 1-, 3-, 6-, and 12-month historical allocation windows across rolling forecast origins. Implementation details are excluded with the company-specific NCLCO hierarchy.


## Rolling backtest with fallback windows (omitted)

The source notebook also compares an actual-proportion approach with 1-to-3-to-12-month fallback windows against independently forecast and rescaled proportions. Company-specific implementation details are not included here.
